# MVP — Predição de Severidade de Ocorrências Aeronáuticas por Tipo de Operação

**Aluna:** Mariane Oliveira  
**Curso:** PUC — Sprint de Machine Learning  
**Dataset:** CENIPA/FAB — Centro de Investigação e Prevenção de Acidentes Aeronáuticos  
**Data:** Junho 2026

---
## Seção 1 — Definição do Problema

### Contexto

O Brasil possui um dos maiores espaços aéreos do mundo. O CENIPA/FAB registra todas as ocorrências aeronáuticas (acidentes e incidentes) desde os anos 2000. Compreender os fatores que determinam a severidade dessas ocorrências é fundamental para políticas de segurança e prevenção.

### Pergunta de Pesquisa

**Principal:** É possível prever a severidade de uma ocorrência aeronáutica com base nas características da aeronave e da operação?

**Secundária:** Quais tipos de operação apresentam maior risco (maior proporção de danos severos)?

### Abordagem Híbrida

O projeto é dividido em duas fases:

- **Fase 1 (Semana 2):** Prever o tipo de operação (`aeronave_tipo_operacao`) a partir das características da ocorrência.
- **Fase 2 (Semana 3):** Para cada tipo de operação, prever a severidade do dano (`aeronave_nivel_dano`).

### Variável-Alvo

- **Fase 1:** `aeronave_tipo_operacao` (REGULAR, TÁXI AÉREO, AGRÍCOLA, PRIVADA, etc.)
- **Fase 2:** `aeronave_nivel_dano` (NENHUM, LEVE, SUBSTANCIAL, DESTRUÍDA)

### Resultado Esperado

Um ranking de risco por tipo de operação, com recomendações regulatórias baseadas em dados.

---
## TODO: REVISAR ESSA SESSAO, MUDEI ALGUNS TARGETS DURANTE O DESENVOLVIMENTO Seção 2 — Descrição dos Dados 

### Fonte

Os dados são públicos e fornecidos pelo **CENIPA/FAB** (Centro de Investigação e Prevenção de Acidentes Aeronáuticos), disponíveis em: https://www.gov.br/transportes/pt-br/assuntos/transporte-aereo/seguranca-de-aviacao/ocorrencias-aeronauticas

### Tabelas Utilizadas

| Arquivo | Descrição | Linhas (~) |
|---|---|---|
| `ocorrencia.csv` | Registro geral das ocorrências (local, data, classificação) | 13.185 |
| `aeronave.csv` | Dados da aeronave envolvida (tipo, operação, fase, dano) | 13.301 |
| `ocorrencia_tipo.csv` | Tipo e categoria da ocorrência (taxonomia ICAO) | 13.900 |
| `fator_contribuinte.csv` | Fatores que contribuíram para a ocorrência | 8.613 |

### Chaves de Junção

- `ocorrencia.codigo_ocorrencia` → `aeronave.codigo_ocorrencia2`
- `ocorrencia.codigo_ocorrencia` → `ocorrencia_tipo.codigo_ocorrencia1`

### Variáveis de Interesse

| Variável | Tabela | Tipo | Papel |
|---|---|---|---|
| `aeronave_tipo_operacao` | aeronave | Categórica | Target Fase 1 |
| `aeronave_nivel_dano` | aeronave | Categórica | Target Fase 2 / Feature |
| `ocorrencia_classificacao` | ocorrencia | Categórica | Feature |
| `aeronave_fase_operacao` | aeronave | Categórica | Feature |
| `aeronave_motor_tipo` | aeronave | Categórica | Feature |
| `aeronave_motor_quantidade` | aeronave | Categórica | Feature |
| `ocorrencia_tipo_categoria` | ocorrencia_tipo | Categórica | Feature |

### 2.1 — Setup: Bibliotecas e Configurações

In [22]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Bibliotecas carregadas com sucesso.')
print(f'NumPy: {np.__version__} | Pandas: {pd.__version__}')

Bibliotecas carregadas com sucesso.
NumPy: 2.4.6 | Pandas: 3.0.3


### 2.2 — Carga dos Dados

In [23]:
# Ajuste o caminho conforme necessário
DATA_PATH = 'data/raw/CENIPA_FAB/'

ocorrencia = pd.read_csv(
    DATA_PATH + 'ocorrencia.csv',
    sep=';',
    encoding='latin-1',
    low_memory=False
)

aeronave = pd.read_csv(
    DATA_PATH + 'aeronave.csv',
    sep=';',
    encoding='latin-1',
    low_memory=False
)

ocorrencia_tipo = pd.read_csv(
    DATA_PATH + 'ocorrencia_tipo.csv',
    sep=';',
    encoding='latin-1',
    low_memory=False
)

fator = pd.read_csv(
    DATA_PATH + 'fator_contribuinte.csv',
    sep=';',
    encoding='latin-1',
    low_memory=False
)

print(f'ocorrencia:      {ocorrencia.shape}')
print(f'aeronave:        {aeronave.shape}')
print(f'ocorrencia_tipo: {ocorrencia_tipo.shape}')
print(f'fator:           {fator.shape}')

ocorrencia:      (13185, 22)
aeronave:        (13301, 23)
ocorrencia_tipo: (13900, 4)
fator:           (8613, 5)


In [24]:
# Junção ocorrencia + aeronave
df = ocorrencia.merge(
    aeronave,
    left_on='codigo_ocorrencia',
    right_on='codigo_ocorrencia2',
    how='inner'
)

# Junção com ocorrencia_tipo (pode haver múltiplos tipos por ocorrência — pegamos o primeiro)
tipo_dedup = ocorrencia_tipo.drop_duplicates(subset='codigo_ocorrencia1', keep='first')
df = df.merge(
    tipo_dedup[['codigo_ocorrencia1', 'ocorrencia_tipo_categoria', 'taxonomia_tipo_icao']],
    left_on='codigo_ocorrencia',
    right_on='codigo_ocorrencia1',
    how='left'
)

print(f'Dataset final: {df.shape[0]} linhas, {df.shape[1]} colunas')
df.head(3)
# df['aeronave_tipo_operacao'].value_counts(dropna=False)
sorted(df['aeronave_nivel_dano'].dropna().unique())


Dataset final: 13301 linhas, 48 colunas


['***', 'DESTRUÍDA', 'LEVE', 'NENHUM', 'SUBSTANCIAL']

---
## Seção 3 — Análise Exploratória dos Dados (EDA)

### 3.1 — Visão Geral do Dataset

In [25]:
print('=== Tipos de dados ===')
print(df.dtypes.value_counts())
print()
print('=== Primeiras linhas ===')
df.head(3)

=== Tipos de dados ===
str        34
int64       9
float64     5
Name: count, dtype: int64

=== Primeiras linhas ===


,codigo_ocorrencia,codigo_ocorrencia1_x,codigo_ocorrencia2_x,codigo_ocorrencia3,codigo_ocorrencia4,ocorrencia_classificacao,ocorrencia_latitude,ocorrencia_longitude,ocorrencia_cidade,ocorrencia_uf,...,aeronave_registro_segmento,aeronave_voo_origem,aeronave_voo_destino,aeronave_fase_operacao,aeronave_tipo_operacao,aeronave_nivel_dano,aeronave_fatalidades_total,codigo_ocorrencia1_y,ocorrencia_tipo_categoria,taxonomia_tipo_icao
0,87125,87125,87125,87125,87125,INCIDENTE,-7.219166666666,-39.26944444444,JUAZEIRO DO NORTE,CE,...,***,VIRACOPOS,ORLANDO BEZERRA DE MENEZES,DESCIDA,REGULAR,NENHUM,0.0,87125,FALHA OU MAU FUNCIONAMENTO DE SISTEMA / COMPON...,SCF-NP
1,87124,87124,87124,87124,87124,INCIDENTE,-18.88361111111,-48.22527777777,UBERLÂNDIA,MG,...,***,TENENTE-CORONEL AVIADOR CÉSAR BOMBONATO,TANCREDO NEVES,SUBIDA,REGULAR,NENHUM,0.0,87124,FALHA OU MAU FUNCIONAMENTO DE SISTEMA / COMPON...,SCF-NP
2,87123,87123,87123,87123,87123,INCIDENTE,-23.43555555555,-46.47305555555,GUARULHOS,SP,...,***,GOVERNADOR ANDRÉ FRANCO MONTORO,PINTO MARTINS,DECOLAGEM,REGULAR,NENHUM,0.0,87123,FALHA OU MAU FUNCIONAMENTO DE SISTEMA / COMPON...,SCF-NP


In [26]:
# Estatísticas das colunas numéricas
df.describe(include='all').T[['count', 'unique', 'top', 'freq', 'mean', 'std', 'min', 'max']]

,count,unique,top,freq,mean,std,min,max
codigo_ocorrencia,13301.0,NaN,NaN,NaN,67809.14638,18131.626733,28256.0,87125.0
codigo_ocorrencia1_x,13301.0,NaN,NaN,NaN,67809.14638,18131.626733,28256.0,87125.0
codigo_ocorrencia2_x,13301.0,NaN,NaN,NaN,67809.14638,18131.626733,28256.0,87125.0
codigo_ocorrencia3,13301.0,NaN,NaN,NaN,67809.14638,18131.626733,28256.0,87125.0
codigo_ocorrencia4,13301.0,NaN,NaN,NaN,67809.14638,18131.626733,28256.0,87125.0
ocorrencia_classificacao,13301,3,INCIDENTE,9252,NaN,NaN,NaN,NaN
ocorrencia_latitude,10525,3721,***,1283,NaN,NaN,NaN,NaN
ocorrencia_longitude,10525,3728,***,1284,NaN,NaN,NaN,NaN
ocorrencia_cidade,13301,1430,RIO DE JANEIRO,926,NaN,NaN,NaN,NaN
ocorrencia_uf,13301,28,SP,3323,NaN,NaN,NaN,NaN


### 3.2 — Análise de Valores Ausentes

O CENIPA usa `***` como placeholder para dados não informados. Precisamos tratar isso como `NaN`.

In [27]:
df.replace(['***', '', 'NULL'], np.nan, inplace=True)

missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_df = missing[missing > 0].reset_index()
missing_df.columns = ['coluna', 'pct_ausente']

fig = px.bar(
    missing_df.head(20),
    x='pct_ausente', y='coluna',
    orientation='h',
    title='Top 20 Colunas com Valores Ausentes (%)',
    labels={'pct_ausente': '% Ausente', 'coluna': ''},
    color='pct_ausente',
    color_continuous_scale='Reds'
)
fig.update_layout(yaxis=dict(autorange='reversed'))
fig.show()

print(f'\nColunas com >50% ausentes: {(missing > 50).sum()}')
print(f'Colunas sem ausentes:      {(missing == 0).sum()}')


Colunas com >50% ausentes: 3
Colunas sem ausentes:      17


### 3.3 — Distribuição da Variável-Alvo (Fase 1): Tipo de Operação

In [28]:
target_counts = df['aeronave_tipo_operacao'].value_counts(dropna=False)
print('Distribuição de aeronave_tipo_operacao:')
print(target_counts)
print(f'\nTotal: {target_counts.sum()} | Classes: {df["aeronave_tipo_operacao"].nunique(dropna=True)}')

Distribuição de aeronave_tipo_operacao:
aeronave_tipo_operacao
REGULAR           4786
PRIVADA           3225
TÁXI AÉREO        1362
INSTRUÇÃO         1167
NaN                989
AGRÍCOLA           840
EXPERIMENTAL       364
POLICIAL           270
ESPECIALIZADA      159
NÃO REGULAR        124
AERODESPORTIVA      15
Name: count, dtype: int64

Total: 13301 | Classes: 10


In [29]:
plot_data = df['aeronave_tipo_operacao'].value_counts().reset_index()
plot_data.columns = ['tipo_operacao', 'count']

fig_bar = px.bar(
    plot_data, x='count', y='tipo_operacao',
    orientation='h',
    title='Ocorrências por Tipo de Operação',
    labels={'count': 'Quantidade', 'tipo_operacao': ''},
    color='count',
    color_continuous_scale='Blues'
)
fig_bar.update_layout(yaxis=dict(autorange='reversed'))
fig_bar.show()

fig_pie = px.pie(
    plot_data, values='count', names='tipo_operacao',
    title='Proporção por Tipo de Operação'
)
fig_pie.show()

print('\nObservação: Há desbalanceamento de classes — será necessário estratificação no split e uso de métricas ponderadas (F1-weighted).')


Observação: Há desbalanceamento de classes — será necessário estratificação no split e uso de métricas ponderadas (F1-weighted).


Observando esse gráfico percebemos que a maior parcela de ocorrencias se encontra na categoria de operação "Regular", esse é um dado a ser lembrado pois proporcionalmente a aviação regular concentra a maioria absoluta de movimentos - pode ser verificado nos dados de VRA da ANAC. Como remover esse viés da avaliação?

### 3.4 — Distribuição da Variável-Alvo (Fase 2): Nível de Dano

In [30]:
dano_counts = df['aeronave_nivel_dano'].value_counts(dropna=False)
print('Distribuição de aeronave_nivel_dano:')
print(dano_counts)

dano_df = dano_counts.reset_index()
dano_df.columns = ['nivel_dano', 'count']

fig = px.bar(
    dano_df, x='nivel_dano', y='count',
    title='Distribuição do Nível de Dano (Target Fase 2)',
    labels={'nivel_dano': 'Nível de Dano', 'count': 'Quantidade'},
    color='count',
    color_continuous_scale='Oranges'
)
fig.show()

Distribuição de aeronave_nivel_dano:
aeronave_nivel_dano
NENHUM         6626
LEVE           3338
SUBSTANCIAL    2416
DESTRUÍDA       548
NaN             373
Name: count, dtype: int64


### 3.5 — Severidade por Tipo de Operação

Esta é a análise central do projeto: qual tipo de operação tem maior proporção de danos graves?

In [31]:
cross = pd.crosstab(
    df['aeronave_tipo_operacao'],
    df['aeronave_nivel_dano'],
    normalize='index'
) * 100

if 'DESTRUÍDA' in cross.columns:
    cross = cross.sort_values('DESTRUÍDA', ascending=False)

cross_long = cross.reset_index().melt(
    id_vars='aeronave_tipo_operacao',
    var_name='nivel_dano',
    value_name='pct'
)

fig = px.bar(
    cross_long,
    x='aeronave_tipo_operacao', y='pct',
    color='nivel_dano',
    barmode='stack',
    title='Distribuição de Dano por Tipo de Operação (%)',
    labels={'aeronave_tipo_operacao': 'Tipo de Operação', 'pct': '%', 'nivel_dano': 'Nível de Dano'},
    color_discrete_sequence=px.colors.diverging.RdYlGn[::-1]
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

print('\n% por tipo de operação:')
cross.round(1)


% por tipo de operação:


aeronave_nivel_dano,DESTRUÍDA,LEVE,NENHUM,SUBSTANCIAL
aeronave_tipo_operacao,,,,
EXPERIMENTAL,17.1,24.0,14.6,44.2
AGRÍCOLA,11.1,17.2,4.3,67.3
PRIVADA,9.0,33.6,26.5,31.0
ESPECIALIZADA,5.8,28.6,39.0,26.6
POLICIAL,3.4,48.1,29.7,18.8
TÁXI AÉREO,3.3,32.6,48.7,15.3
INSTRUÇÃO,2.3,35.0,32.5,30.2
NÃO REGULAR,1.6,31.7,51.2,15.4
REGULAR,0.1,20.1,79.2,0.6


### 3.6 — Análise das Features Candidatas

In [32]:
fase_counts = df['aeronave_fase_operacao'].value_counts().head(10).reset_index()
fase_counts.columns = ['fase', 'count']

motor_counts = df['aeronave_motor_tipo'].value_counts().head(8).reset_index()
motor_counts.columns = ['motor', 'count']

class_counts = df['ocorrencia_classificacao'].value_counts().reset_index()
class_counts.columns = ['classificacao', 'count']

fig1 = px.bar(
    fase_counts, x='count', y='fase', orientation='h',
    title='Top 10 Fases de Operação',
    labels={'count': 'Quantidade', 'fase': ''},
    color='count', color_continuous_scale='Blues'
)
fig1.update_layout(yaxis=dict(autorange='reversed'))
fig1.show()

fig2 = px.bar(
    motor_counts, x='count', y='motor', orientation='h',
    title='Tipo de Motor',
    labels={'count': 'Quantidade', 'motor': ''},
    color='count', color_continuous_scale='Greens'
)
fig2.update_layout(yaxis=dict(autorange='reversed'))
fig2.show()

fig3 = px.bar(
    class_counts, x='count', y='classificacao', orientation='h',
    title='Classificação da Ocorrência',
    labels={'count': 'Quantidade', 'classificacao': ''},
    color='count', color_continuous_scale='Oranges'
)
fig3.update_layout(yaxis=dict(autorange='reversed'))
fig3.show()

### 3.7 — Análise Temporal

In [33]:
df['ocorrencia_dia'] = pd.to_datetime(df['ocorrencia_dia'], format='%d/%m/%Y', errors='coerce')
df['ano'] = df['ocorrencia_dia'].dt.year

por_ano = df.groupby('ano').size().reset_index(name='total')
por_ano = por_ano[por_ano['ano'].between(2010, 2025)]

fig = px.line(
    por_ano, x='ano', y='total',
    markers=True,
    title='Ocorrências por Ano (2010–2025)',
    labels={'ano': 'Ano', 'total': 'Total de Ocorrências'},
    color_discrete_sequence=['steelblue']
)
fig.update_xaxes(dtick=1)
fig.show()

---
## Seção 4 — Preparação dos Dados

### 4.1 — Seleção de Features e Targets

Baseado na EDA, selecionamos features com boa cobertura e relevância para o problema.

In [34]:
FEATURES = [
    'ocorrencia_classificacao',
    'aeronave_fase_operacao',
    'aeronave_motor_tipo',
    'aeronave_motor_quantidade',
    'aeronave_nivel_dano',
    'ocorrencia_tipo_categoria',
]

TARGET_FASE1 = 'aeronave_tipo_operacao'
TARGET_FASE2 = 'aeronave_nivel_dano'

df_model = df[FEATURES + [TARGET_FASE1]].copy()

print(f'Shape antes de limpeza: {df_model.shape}')
print(f'\nNaN por coluna:')
print(df_model.isnull().sum().sort_values(ascending=False))

Shape antes de limpeza: (13301, 7)

NaN por coluna:
aeronave_motor_tipo          1267
aeronave_tipo_operacao        989
aeronave_motor_quantidade     567
aeronave_nivel_dano           373
aeronave_fase_operacao         30
ocorrencia_tipo_categoria       7
ocorrencia_classificacao        0
dtype: int64


### 4.2 — Limpeza e Tratamento de Missing Values

In [35]:
# Remover linhas onde o TARGET é nulo (não há o que prever)
df_model = df_model.dropna(subset=[TARGET_FASE1])

# Remover classes muito raras (menos de 30 exemplos) — inviáveis para treino
min_samples = 30
class_counts = df_model[TARGET_FASE1].value_counts()
classes_validas = class_counts[class_counts >= min_samples].index
df_model = df_model[df_model[TARGET_FASE1].isin(classes_validas)]

print(f'Shape após limpeza: {df_model.shape}')
print(f'\nClasses mantidas ({len(classes_validas)}):')
print(df_model[TARGET_FASE1].value_counts())

Shape após limpeza: (12297, 7)

Classes mantidas (9):
aeronave_tipo_operacao
REGULAR          4786
PRIVADA          3225
TÁXI AÉREO       1362
INSTRUÇÃO        1167
AGRÍCOLA          840
EXPERIMENTAL      364
POLICIAL          270
ESPECIALIZADA     159
NÃO REGULAR       124
Name: count, dtype: int64


### 4.3 — Encoding de Variáveis Categóricas

Usamos `OneHotEncoding` para variáveis categóricas nominais, evitando ordinalidade artificial. Valores ausentes são imputados pela moda dentro do pipeline, após o split, prevenindo data leakage.

In [36]:
FEAT_CAT = [
    'ocorrencia_classificacao',
    'aeronave_fase_operacao',
    'aeronave_motor_tipo',
    'aeronave_motor_quantidade',
    'aeronave_nivel_dano',
    'ocorrencia_tipo_categoria',
]

X = df_model[FEAT_CAT].copy()
y = df_model[TARGET_FASE1].copy()

print(f'X: {X.shape} | y: {y.shape}')
print(f'Classes: {sorted(y.unique())}')

X: (12297, 6) | y: (12297,)
Classes: ['AGRÍCOLA', 'ESPECIALIZADA', 'EXPERIMENTAL', 'INSTRUÇÃO', 'NÃO REGULAR', 'POLICIAL', 'PRIVADA', 'REGULAR', 'TÁXI AÉREO']


### 4.4 — Divisão Treino / Teste

**Importante:** O split é feito ANTES do pré-processamento para evitar data leakage. Usamos `stratify=y` para manter a proporção de classes.

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f'Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras')
print(f'Proporção treino/teste: {X_train.shape[0]/len(X)*100:.0f}% / {X_test.shape[0]/len(X)*100:.0f}%')
print()
print('Distribuição de classes no treino:')
print(y_train.value_counts(normalize=True).round(3))

Treino: 9837 amostras | Teste: 2460 amostras
Proporção treino/teste: 80% / 20%

Distribuição de classes no treino:
aeronave_tipo_operacao
REGULAR          0.389
PRIVADA          0.262
TÁXI AÉREO       0.111
INSTRUÇÃO        0.095
AGRÍCOLA         0.068
EXPERIMENTAL     0.030
POLICIAL         0.022
ESPECIALIZADA    0.013
NÃO REGULAR      0.010
Name: proportion, dtype: float64


### 4.5 — Pipeline de Pré-processamento

O pipeline é ajustado (`fit`) apenas nos dados de treino e aplicado (`transform`) no teste. Isso previne data leakage.

In [38]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('cat', cat_transformer, FEAT_CAT),
])

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep  = preprocessor.transform(X_test)

print(f'X_train pré-processado: {X_train_prep.shape}')
print(f'X_test  pré-processado: {X_test_prep.shape}')
print('\n✅ Pipeline ajustado no treino e aplicado ao teste — sem data leakage.')

X_train pré-processado: (9837, 132)
X_test  pré-processado: (2460, 132)

✅ Pipeline ajustado no treino e aplicado ao teste — sem data leakage.


### 4.6 — Encoding do Target

In [39]:
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

print('Mapeamento de classes:')
for i, cls in enumerate(le.classes_):
    print(f'  {i} → {cls}')

Mapeamento de classes:
  0 → AGRÍCOLA
  1 → ESPECIALIZADA
  2 → EXPERIMENTAL
  3 → INSTRUÇÃO
  4 → NÃO REGULAR
  5 → POLICIAL
  6 → PRIVADA
  7 → REGULAR
  8 → TÁXI AÉREO


### 4.7 — Resumo da Preparação

| Etapa | Decisão | Justificativa |
|---|---|---|
| Missing values categóricos | Imputação pela moda | Preserva distribuição das classes |
| Encoding categórico | OneHotEncoding | Evita ordinalidade artificial |
| Classes raras (<30) | Removidas | Inviáveis para treino/avaliação |
| Split | 80/20 estratificado | Mantém proporção de classes |
| Ordem de operações | Split → Fit → Transform | Previne data leakage |

**Próximo passo (Seção 5):** Treinar os modelos de Fase 1 — Regressão Logística, Random Forest, XGBoost.

---
## Seção 5 — Modelagem (Fase 1): Previsão do Tipo de Operação

Seguindo a ordem de construção recomendada pelo plano de implementação:

1. **Baseline — Regressão Logística:** referência mínima de desempenho, modelo linear e interpretável
2. **Modelo 1 — Random Forest:** ensemble de árvores, robusto sem normalização prévia
3. **Modelo 2 — XGBoost:** gradient boosting, maior poder preditivo

Todos os modelos compartilham `X_train_prep` / `X_test_prep` e `y_train_enc` / `y_test_enc` para comparação justa.
CV de 5-fold é aplicado nos dois primeiros modelos para estimar variância do desempenho; omitido no XGBoost pelo custo computacional com features OHE densas.

### 5.1 — Baseline: Regressão Logística

In [40]:
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_prep, y_train_enc)
y_pred_lr = lr.predict(X_test_prep)

cv_lr = cross_val_score(lr, X_train_prep, y_train_enc, cv=5, scoring='f1_weighted', n_jobs=-1)

print('=== Baseline: Regressão Logística ===')
print(f'Acurácia (teste):    {accuracy_score(y_test_enc, y_pred_lr):.4f}')
print(f'F1 weighted (teste): {f1_score(y_test_enc, y_pred_lr, average="weighted"):.4f}')
print(f'F1 CV 5-fold:        {cv_lr.mean():.4f} ± {cv_lr.std():.4f}')

=== Baseline: Regressão Logística ===
Acurácia (teste):    0.6817
F1 weighted (teste): 0.6553
F1 CV 5-fold:        0.6405 ± 0.0107


### 5.2 — Modelo 1: Random Forest

In [41]:
rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train_prep, y_train_enc)
y_pred_rf = rf.predict(X_test_prep)

cv_rf = cross_val_score(rf, X_train_prep, y_train_enc, cv=5, scoring='f1_weighted', n_jobs=-1)

print('=== Modelo 1: Random Forest ===')
print(f'Acurácia (teste):    {accuracy_score(y_test_enc, y_pred_rf):.4f}')
print(f'F1 weighted (teste): {f1_score(y_test_enc, y_pred_rf, average="weighted"):.4f}')
print(f'F1 CV 5-fold:        {cv_rf.mean():.4f} ± {cv_rf.std():.4f}')

=== Modelo 1: Random Forest ===
Acurácia (teste):    0.6557
F1 weighted (teste): 0.6377
F1 CV 5-fold:        0.6298 ± 0.0105


### 5.3 — Modelo 2: XGBoost

In [42]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(random_state=RANDOM_STATE, eval_metric='mlogloss', verbosity=0)
xgb_model.fit(X_train_prep, y_train_enc)
y_pred_xgb = xgb_model.predict(X_test_prep)

print('=== Modelo 2: XGBoost ===')
print(f'Acurácia (teste):    {accuracy_score(y_test_enc, y_pred_xgb):.4f}')
print(f'F1 weighted (teste): {f1_score(y_test_enc, y_pred_xgb, average="weighted"):.4f}')
print('(CV omitido — custo computacional elevado com features OHE densas)')

=== Modelo 2: XGBoost ===
Acurácia (teste):    0.6833
F1 weighted (teste): 0.6608
(CV omitido — custo computacional elevado com features OHE densas)


### 5.4 — Validação Cruzada Comparativa (5-Fold Estratificado)

Após treinar os três modelos, executamos a validação cruzada com `StratifiedKFold` para:
- Estimar a **estabilidade** do desempenho (variância entre folds)
- Confirmar se os resultados no conjunto de teste são representativos
- Identificar modelos com overfitting (alta variância entre folds)

Métrica: **F1 weighted** — adequada para o desbalanceamento de classes presente no dataset.

In [50]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_results = []
for model, name in [
    (lr,        'Regressão Logística'),
    (rf,        'Random Forest'),
    (xgb_model, 'XGBoost'),
]:
    scores = cross_val_score(
        model, X_train_prep, y_train_enc,
        cv=skf, scoring='f1_weighted', n_jobs=-1
    )
    for fold, score in enumerate(scores, 1):
        cv_results.append({'Modelo': name, 'Fold': fold, 'F1 weighted': score})

cv_df = pd.DataFrame(cv_results)

print('=== Validação Cruzada 5-Fold — F1 Weighted ===')
print(cv_df.groupby('Modelo')['F1 weighted'].agg(['mean', 'std']).round(4).to_string())

fig = px.box(
    cv_df, x='Modelo', y='F1 weighted',
    points='all',
    color='Modelo',
    title='Distribuição F1 Weighted por Fold — Comparação de Modelos',
    labels={'F1 weighted': 'F1 (weighted)', 'Modelo': ''}
)
fig.update_layout(showlegend=False, yaxis_range=[0, 1])
fig.show()

=== Validação Cruzada 5-Fold — F1 Weighted ===
                       mean     std
Modelo                             
Random Forest        0.6335  0.0123
Regressão Logística  0.6423  0.0071
XGBoost              0.6448  0.0095


---
## Seção 6 — Avaliação (Fase 1)

Comparação dos três modelos nas mesmas métricas, seguida de análise detalhada do melhor desempenho via matriz de confusão, importância de features e relatório de classificação.

### 6.1 — Tabela Comparativa de Modelos

In [47]:
from sklearn.metrics import precision_score, recall_score

results = pd.DataFrame({
    'Modelo': ['Regressão Logística', 'Random Forest', 'XGBoost'],
    'Acurácia': [
        accuracy_score(y_test_enc, y_pred_lr),
        accuracy_score(y_test_enc, y_pred_rf),
        accuracy_score(y_test_enc, y_pred_xgb),
    ],
    'Precisão (w)': [
        precision_score(y_test_enc, y_pred_lr, average='weighted', zero_division=0),
        precision_score(y_test_enc, y_pred_rf, average='weighted', zero_division=0),
        precision_score(y_test_enc, y_pred_xgb, average='weighted', zero_division=0),
    ],
    'Recall (w)': [
        recall_score(y_test_enc, y_pred_lr, average='weighted', zero_division=0),
        recall_score(y_test_enc, y_pred_rf, average='weighted', zero_division=0),
        recall_score(y_test_enc, y_pred_xgb, average='weighted', zero_division=0),
    ],
    'F1 (w)': [
        f1_score(y_test_enc, y_pred_lr, average='weighted', zero_division=0),
        f1_score(y_test_enc, y_pred_rf, average='weighted', zero_division=0),
        f1_score(y_test_enc, y_pred_xgb, average='weighted', zero_division=0),
    ],
})

print(results.round(4).to_string(index=False))

fig = px.bar(
    results.melt(id_vars='Modelo', var_name='Métrica', value_name='Valor'),
    x='Modelo', y='Valor', color='Métrica', barmode='group',
    title='Comparação de Modelos — Fase 1',
    labels={'Valor': 'Score', 'Modelo': ''},
    text_auto='.3f'
)
fig.update_layout(yaxis_range=[0, 1])
fig.show()

             Modelo  Acurácia  Precisão (w)  Recall (w)  F1 (w)
Regressão Logística    0.6817        0.6650      0.6817  0.6553
      Random Forest    0.6557        0.6274      0.6557  0.6377
            XGBoost    0.6833        0.6600      0.6833  0.6608


### 6.2 — Matrizes de Confusão

In [48]:
class_names = le.classes_

for y_pred, model_name in [
    (y_pred_lr, 'Regressão Logística'),
    (y_pred_rf, 'Random Forest'),
    (y_pred_xgb, 'XGBoost'),
]:
    cm = confusion_matrix(y_test_enc, y_pred)
    fig = px.imshow(
        cm,
        labels=dict(x='Predito', y='Real', color='Contagem'),
        x=list(class_names),
        y=list(class_names),
        text_auto=True,
        color_continuous_scale='Blues',
        title=f'Matriz de Confusão — {model_name}'
    )
    fig.update_xaxes(tickangle=-45)
    fig.show()

### 6.3 — Importância de Features (Random Forest)

In [49]:
feature_names = preprocessor.get_feature_names_out()
feat_imp = pd.DataFrame({
    'feature': feature_names,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False).head(20)

fig = px.bar(
    feat_imp, x='importance', y='feature',
    orientation='h',
    title='Top 20 Features por Importância — Random Forest',
    labels={'importance': 'Importância', 'feature': ''},
    color='importance',
    color_continuous_scale='Blues'
)
fig.update_layout(yaxis=dict(autorange='reversed'))
fig.show()

### 6.4 — Relatório Detalhado (Random Forest)

In [46]:
print('=== Classification Report — Random Forest ===')
print(classification_report(y_test_enc, y_pred_rf, target_names=le.classes_))

=== Classification Report — Random Forest ===
               precision    recall  f1-score   support

     AGRÍCOLA       0.60      0.54      0.57       168
ESPECIALIZADA       0.10      0.03      0.05        32
 EXPERIMENTAL       0.08      0.03      0.04        73
    INSTRUÇÃO       0.50      0.44      0.47       233
  NÃO REGULAR       0.00      0.00      0.00        25
     POLICIAL       0.49      0.41      0.44        54
      PRIVADA       0.53      0.65      0.59       645
      REGULAR       0.85      0.91      0.88       957
   TÁXI AÉREO       0.47      0.36      0.41       273

     accuracy                           0.66      2460
    macro avg       0.40      0.37      0.38      2460
 weighted avg       0.63      0.66      0.64      2460



---
## Seção 7 — Otimização de Hiperparâmetros

> 🔜 **A ser desenvolvida na Semana 3** — GridSearchCV no melhor modelo

---
## Seção 8 — Modelagem (Fase 2): Severidade por Tipo de Operação

> 🔜 **A ser desenvolvida na Semana 3** — Modelos separados por tipo de operação

---
## Seção 9 — Avaliação (Fase 2)

> 🔜 **A ser desenvolvida na Semana 3** — Ranking de risco por tipo de operação

---
## Seção 10 — Conclusão

> 🔜 **A ser desenvolvida na Semana 4** — Achados, limitações, próximos passos